# 05 — SQL Analysis

## UK Data Analyst Job Market Intelligence

This notebook analyses the processed UK analytics job-market datasets using
SQL with DuckDB.

The objective is to reproduce and extend key business analyses using SQL,
including:

- market composition;
- salary benchmarking;
- geographic demand;
- advertiser concentration;
- job-family comparisons;
- technology and capability mentions;
- ranking and window-function analysis.

SQL queries are applied to the same cleaned analytical datasets used in the
Python workflow, demonstrating a reproducible multi-tool analytics pipeline.

## 1. Environment Setup

DuckDB is used as the SQL engine for this notebook.

The processed CSV files created during the Python workflow are queried through
SQL so that the same analytical dataset can be examined using relational
techniques including joins, aggregations, CTEs and window functions.

1.1 Instalar/importar DuckDB

In [1]:
%pip install -q duckdb

In [2]:
import duckdb
import pandas as pd
from google.colab import drive

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
PROJECT_PATH = "/content/drive/MyDrive/UK_Data_Analyst_Market"

PROCESSED_PATH = f"{PROJECT_PATH}/data/processed"
EDA_PATH = f"{PROJECT_PATH}/data/eda_outputs"
SKILLS_PATH = f"{PROJECT_PATH}/data/skills_outputs"

1.2 Criar a conexão SQL

In [5]:
con = duckdb.connect()

In [6]:
con.execute(f"""
CREATE OR REPLACE TABLE vacancies AS
SELECT *
FROM read_csv_auto(
    '{PROCESSED_PATH}/uk_jobs_classified.csv',
    HEADER = TRUE
);
""")

In [7]:
con.execute(f"""
CREATE OR REPLACE TABLE core_jobs AS
SELECT *
FROM read_csv_auto(
    '{PROCESSED_PATH}/uk_core_analytics_jobs.csv',
    HEADER = TRUE
);
""")

In [8]:
con.execute(f"""
CREATE OR REPLACE TABLE extended_jobs AS
SELECT *
FROM read_csv_auto(
    '{PROCESSED_PATH}/uk_extended_analytics_jobs.csv',
    HEADER = TRUE
);
""")

In [9]:
 con.execute(f"""
CREATE OR REPLACE TABLE job_search_terms AS
SELECT *
FROM read_csv_auto(
    '{PROCESSED_PATH}/job_search_terms.csv',
    HEADER = TRUE
);
""")

## 2. SQL Data Validation

The first SQL queries validate that the processed datasets were loaded
correctly and retain the expected record counts and unique identifiers.

In [10]:
con.sql("""
SELECT
    COUNT(*) AS total_vacancies,
    COUNT(DISTINCT job_id) AS unique_job_ids
FROM vacancies;
""").df()

,total_vacancies,unique_job_ids
0,390,390


In [11]:
con.sql("""
SELECT
    COUNT(*) AS core_vacancies,
    COUNT(DISTINCT job_id) AS unique_core_jobs
FROM core_jobs;
""").df()

,core_vacancies,unique_core_jobs
0,244,244


In [12]:
con.sql("""
SELECT
    COUNT(*) AS extended_vacancies,
    COUNT(DISTINCT job_id) AS unique_extended_jobs
FROM extended_jobs;
""").df()

,extended_vacancies,unique_extended_jobs
0,334,334


2.1 Ver o schema pelo SQL

In [13]:
con.sql("""
DESCRIBE core_jobs;
""").df()

,column_name,column_type,null,key,default,extra
0,job_id,BIGINT,YES,None,None,None
1,title,VARCHAR,YES,None,None,None
2,company,VARCHAR,YES,None,None,None
3,location,VARCHAR,YES,None,None,None
4,created,TIMESTAMP WITH TIME ZONE,YES,None,None,None
5,description,VARCHAR,YES,None,None,None
6,salary_min,DOUBLE,YES,None,None,None
7,salary_max,DOUBLE,YES,None,None,None
8,salary_is_predicted,BIGINT,YES,None,None,None
9,latitude,DOUBLE,YES,None,None,None


In [14]:
con.sql("""
SELECT
    job_id,
    title,
    company,
    job_family,
    seniority,
    salary_midpoint
FROM core_jobs
LIMIT 10;
""").df()

,job_id,title,company,job_family,seniority,salary_midpoint
0,5840324082,Lead Data Analyst/Project Controller,BAE Systems,Data Analyst,Lead / Principal,53889.32
1,5840324089,Lead Data Analyst/Project Controller,BAE Systems,Data Analyst,Lead / Principal,53416.18
2,5840324100,Lead Data Analyst/Project Controller,BAE Systems,Data Analyst,Lead / Principal,63011.66
3,5835920078,Data Analyst,VANRATH,Data Analyst,Unspecified,40000.00
4,5829131501,Data Analyst,MCS Group,Data Analyst,Unspecified,30000.00
5,5823507727,Data Analyst,Ofcom,Data Analyst,Unspecified,26645.01
6,5818756002,Master Data Analyst,MCS Group,Data Analyst,Unspecified,40000.00
7,5818755973,Data Analyst - Manufacturing,VANRATH,Data Analyst,Unspecified,30000.00
8,5840658196,Lead Data Analyst/Project Controller,BAE Systems,Data Analyst,Lead / Principal,56400.93
9,5812354048,Data Analyst,Sellick Partnership,Data Analyst,Unspecified,46500.00


2.2 Primeira query analítica

In [15]:
con.sql("""
SELECT
    job_family,
    COUNT(*) AS vacancies,
    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct
FROM core_jobs
GROUP BY job_family
ORDER BY vacancies DESC;
""").df()

,job_family,vacancies,share_pct
0,Data Analyst,143,58.6
1,Reporting Analytics,46,18.9
2,Business Intelligence,46,18.9
3,Insights Analytics,7,2.9
4,Management Information,2,0.8


2.3 Validação de escopo ocupacional

In [16]:
con.sql("""
SELECT
    occupational_scope,
    COUNT(*) AS vacancies,
    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct
FROM vacancies
GROUP BY occupational_scope
ORDER BY vacancies DESC;
""").df()

,occupational_scope,vacancies,share_pct
0,core_analytics,244,62.6
1,adjacent_analytics,90,23.1
2,exclude_occupation,56,14.4


In [17]:
con.sql("""
SELECT
    occupational_scope,
    COUNT(*) AS vacancies,
    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct
FROM vacancies
GROUP BY occupational_scope
ORDER BY vacancies DESC;
""").df()

,occupational_scope,vacancies,share_pct
0,core_analytics,244,62.6
1,adjacent_analytics,90,23.1
2,exclude_occupation,56,14.4


## 3. Salary Analysis with SQL

This section uses SQL aggregations, conditional filtering and statistical
functions to benchmark advertised salaries across job families, geographic
scope and employment structure.

Adzuna-predicted salaries are separated from explicitly advertised salaries
to preserve comparability with the previous analytical workflow.

In [18]:
con.sql("""
SELECT
    salary_source,
    COUNT(*) AS vacancies,
    ROUND(AVG(salary_midpoint), 2) AS mean_salary,
    ROUND(MEDIAN(salary_midpoint), 2) AS median_salary,
    ROUND(MIN(salary_midpoint), 2) AS min_salary,
    ROUND(MAX(salary_midpoint), 2) AS max_salary
FROM core_jobs
GROUP BY salary_source
ORDER BY vacancies DESC;
""").df()

,salary_source,vacancies,mean_salary,median_salary,min_salary,max_salary
0,Adzuna predicted,124,49786.31,49106.49,26645.01,76555.15
1,Advertised / non-predicted,120,51868.83,44750.00,19200.00,136500.00


### 3.1 Geographic Salary Comparison

Geographic scope is derived directly in SQL using conditional logic.

Locations explicitly containing "London" are classified as London, records
reported only as "UK" are treated as geographically unspecified, and all
remaining explicitly localised vacancies are grouped as Rest of UK.

Only explicitly advertised salaries are used for the salary comparison.

In [19]:
con.sql("""
SELECT
    CASE
        WHEN LOWER(location) LIKE '%london%'
            THEN 'London'

        WHEN LOWER(TRIM(location)) = 'uk'
            THEN 'UK-wide / Unspecified'

        ELSE 'Rest of UK'
    END AS geo_scope,

    COUNT(*) AS vacancies,

    ROUND(
        AVG(salary_midpoint),
        2
    ) AS mean_salary,

    ROUND(
        MEDIAN(salary_midpoint),
        2
    ) AS median_salary

FROM core_jobs

WHERE salary_is_predicted = 0

GROUP BY geo_scope

ORDER BY vacancies DESC;
""").df()

,geo_scope,vacancies,mean_salary,median_salary
0,Rest of UK,86,48983.17,40053.0
1,London,31,61467.73,52500.0
2,UK-wide / Unspecified,3,35402.33,39507.0


3.2 Agora calcule o London Premium inteiramente em SQL

In [20]:
con.sql("""
WITH geo_salary AS (

    SELECT
        CASE
            WHEN LOWER(location) LIKE '%london%'
                THEN 'London'

            WHEN LOWER(TRIM(location)) = 'uk'
                THEN 'UK-wide / Unspecified'

            ELSE 'Rest of UK'
        END AS geo_scope,

        salary_midpoint

    FROM core_jobs

    WHERE salary_is_predicted = 0
),

median_by_geo AS (

    SELECT
        geo_scope,
        MEDIAN(salary_midpoint) AS median_salary

    FROM geo_salary

    WHERE geo_scope IN (
        'London',
        'Rest of UK'
    )

    GROUP BY geo_scope
)

SELECT
    MAX(
        CASE
            WHEN geo_scope = 'London'
            THEN median_salary
        END
    ) AS london_median,

    MAX(
        CASE
            WHEN geo_scope = 'Rest of UK'
            THEN median_salary
        END
    ) AS rest_uk_median,

    ROUND(
        (
            MAX(
                CASE
                    WHEN geo_scope = 'London'
                    THEN median_salary
                END
            )
            -
            MAX(
                CASE
                    WHEN geo_scope = 'Rest of UK'
                    THEN median_salary
                END
            )
        )
        /
        MAX(
            CASE
                WHEN geo_scope = 'Rest of UK'
                THEN median_salary
            END
        )
        * 100,
        1
    ) AS london_premium_pct

FROM median_by_geo;
""").df()

,london_median,rest_uk_median,london_premium_pct
0,52500.0,40053.0,31.1


3.3 Salary por Job Family em SQL

In [21]:
con.sql("""
SELECT
    job_family,

    COUNT(*) AS advertised_vacancies,

    ROUND(
        AVG(salary_midpoint),
        2
    ) AS mean_salary,

    ROUND(
        MEDIAN(salary_midpoint),
        2
    ) AS median_salary,

    ROUND(
        MIN(salary_midpoint),
        2
    ) AS min_salary,

    ROUND(
        MAX(salary_midpoint),
        2
    ) AS max_salary

FROM core_jobs

WHERE salary_is_predicted = 0

GROUP BY job_family

ORDER BY median_salary DESC;
""").df()

,job_family,advertised_vacancies,mean_salary,median_salary,min_salary,max_salary
0,Management Information,1,50000.00,50000.0,50000.0,50000.0
1,Insights Analytics,5,50489.20,50000.0,38480.0,60000.0
2,Data Analyst,82,55058.69,45000.0,22880.0,136500.0
3,Reporting Analytics,10,48970.00,43500.0,19200.0,92500.0
4,Business Intelligence,22,41695.50,38728.5,28000.0,80000.0


In [22]:
con.sql("""
SELECT
    job_family,

    COUNT(*) AS advertised_vacancies,

    ROUND(
        MEDIAN(salary_midpoint),
        2
    ) AS median_salary

FROM core_jobs

WHERE salary_is_predicted = 0

GROUP BY job_family

HAVING COUNT(*) >= 10

ORDER BY median_salary DESC;
""").df()

,job_family,advertised_vacancies,median_salary
0,Data Analyst,82,45000.0
1,Reporting Analytics,10,43500.0
2,Business Intelligence,22,38728.5


## 4. Window Functions and Ranking

Window functions are used to rank vacancies within analytical groups without
collapsing individual records.

This allows salary comparisons to retain row-level job information while
adding relative position within job families and geographic segments.

In [23]:
con.sql("""
SELECT
    job_id,
    title,
    company,
    job_family,
    salary_midpoint,

    RANK() OVER (
        PARTITION BY job_family
        ORDER BY salary_midpoint DESC
    ) AS salary_rank_within_family

FROM core_jobs

WHERE salary_is_predicted = 0

ORDER BY
    job_family,
    salary_rank_within_family;
""").df()

,job_id,title,company,job_family,salary_midpoint,salary_rank_within_family
0,5843030453,Sr Business Intelligence Analyst - Commercial ...,Palo Alto Networks,Business Intelligence,80000.0,1
1,5815905845,"Business Intelligence Analyst (Finance, SAP, P...",Eriban Business Services Ltd,Business Intelligence,60000.0,2
2,5846995508,Lead BI Analyst - Tableau,Harnham - Data & Analytics Recruitment,Business Intelligence,56000.0,3
3,5842064165,Lead Business Intelligence Analyst,Harnham - Data & Analytics Recruitment,Business Intelligence,55000.0,4
4,5829404077,Business Intelligence Analyst,Datatech Analytics,Business Intelligence,50000.0,5
...,...,...,...,...,...,...
115,5840202780,ERP and Reporting Analyst,Erin Associates,Reporting Analytics,42500.0,6
116,5838618941,ERP and Reporting Analyst,Erin Associates,Reporting Analytics,42500.0,6
117,5840649681,Data & Reporting Analyst,Robert Half,Reporting Analytics,37500.0,8
118,5842137733,Data & Reporting Analyst,Robert Half,Reporting Analytics,35000.0,9


In [24]:
con.sql("""
WITH ranked_jobs AS (

    SELECT
        job_id,
        title,
        company,
        job_family,
        salary_midpoint,

        ROW_NUMBER() OVER (
            PARTITION BY job_family
            ORDER BY salary_midpoint DESC
        ) AS salary_rank

    FROM core_jobs

    WHERE salary_is_predicted = 0
)

SELECT
    job_family,
    salary_rank,
    title,
    company,
    ROUND(
        salary_midpoint,
        2
    ) AS salary_midpoint

FROM ranked_jobs

WHERE salary_rank <= 3

ORDER BY
    job_family,
    salary_rank;
""").df()

,job_family,salary_rank,title,company,salary_midpoint
0,Business Intelligence,1,Sr Business Intelligence Analyst - Commercial ...,Palo Alto Networks,80000.0
1,Business Intelligence,2,"Business Intelligence Analyst (Finance, SAP, P...",Eriban Business Services Ltd,60000.0
2,Business Intelligence,3,Lead BI Analyst - Tableau,Harnham - Data & Analytics Recruitment,56000.0
3,Data Analyst,1,Data Analyst- Supply Chain,Sanderson,136500.0
4,Data Analyst,2,Data Analyst,TXP,124280.0
5,Data Analyst,3,Data Analyst,TXP Technology x People,124280.0
6,Insights Analytics,1,EMEA Customer & Marketing Insights Lead,JSL Solutions Ltd,60000.0
7,Insights Analytics,2,Senior Reporting and Insights Analyst,Clear Business,57000.0
8,Insights Analytics,3,Data Reporting and Insights Analyst,NFU Mutual,50000.0
9,Management Information,1,"HR MI, Systems & Analytics Analyst",NRG,50000.0


In [25]:
con.sql("""
SELECT
    title,
    job_family,
    salary_midpoint,

    ROW_NUMBER() OVER (
        PARTITION BY job_family
        ORDER BY salary_midpoint DESC
    ) AS row_number_rank,

    RANK() OVER (
        PARTITION BY job_family
        ORDER BY salary_midpoint DESC
    ) AS rank_with_ties,

    DENSE_RANK() OVER (
        PARTITION BY job_family
        ORDER BY salary_midpoint DESC
    ) AS dense_rank

FROM core_jobs

WHERE
    salary_is_predicted = 0
    AND job_family = 'Data Analyst'

ORDER BY salary_midpoint DESC

LIMIT 20;
""").df()

,title,job_family,salary_midpoint,row_number_rank,rank_with_ties,dense_rank
0,Data Analyst- Supply Chain,Data Analyst,136500.0,1,1,1
1,Data Analyst,Data Analyst,124280.0,2,2,2
2,Data Analyst,Data Analyst,124280.0,3,2,2
3,Data Analyst,Data Analyst,123500.0,4,4,3
4,Data Analyst,Data Analyst,113750.0,5,5,4
5,Data Analyst,Data Analyst,107250.0,6,6,5
6,Data Analyst,Data Analyst,101400.0,7,7,6
7,Systems Data Analyst – SQL / NoSQL / Data Migr...,Data Analyst,100100.0,8,8,7
8,Data Analyst,Data Analyst,100000.0,9,9,8
9,Data Analyst,Data Analyst,100000.0,10,9,8


## 5. Salary Bands and Conditional Aggregation

Salary bands are derived using SQL `CASE WHEN` logic to examine the
distribution of explicitly advertised salaries.

Conditional aggregation is then used to summarise multiple salary segments
within a single query.

In [26]:
con.sql("""
SELECT
    CASE
        WHEN salary_midpoint < 30000
            THEN 'Under £30k'

        WHEN salary_midpoint < 40000
            THEN '£30k–£39,999'

        WHEN salary_midpoint < 50000
            THEN '£40k–£49,999'

        WHEN salary_midpoint < 60000
            THEN '£50k–£59,999'

        WHEN salary_midpoint < 80000
            THEN '£60k–£79,999'

        WHEN salary_midpoint < 100000
            THEN '£80k–£99,999'

        ELSE '£100k+'
    END AS salary_band,

    COUNT(*) AS vacancies

FROM core_jobs

WHERE salary_is_predicted = 0

GROUP BY salary_band

ORDER BY
    MIN(salary_midpoint);
""").df()

,salary_band,vacancies
0,Under £30k,8
1,"£30k–£39,999",38
2,"£40k–£49,999",28
3,"£50k–£59,999",18
4,"£60k–£79,999",8
5,"£80k–£99,999",10
6,£100k+,10


5.2 Acrescentar percentual usando window function

In [27]:
con.sql("""
WITH salary_bands AS (

    SELECT
        CASE
            WHEN salary_midpoint < 30000
                THEN 'Under £30k'

            WHEN salary_midpoint < 40000
                THEN '£30k–£39,999'

            WHEN salary_midpoint < 50000
                THEN '£40k–£49,999'

            WHEN salary_midpoint < 60000
                THEN '£50k–£59,999'

            WHEN salary_midpoint < 80000
                THEN '£60k–£79,999'

            WHEN salary_midpoint < 100000
                THEN '£80k–£99,999'

            ELSE '£100k+'
        END AS salary_band,

        salary_midpoint

    FROM core_jobs

    WHERE salary_is_predicted = 0
)

SELECT
    salary_band,
    COUNT(*) AS vacancies,

    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct

FROM salary_bands

GROUP BY salary_band

ORDER BY
    MIN(salary_midpoint);
""").df()

,salary_band,vacancies,share_pct
0,Under £30k,8,6.7
1,"£30k–£39,999",38,31.7
2,"£40k–£49,999",28,23.3
3,"£50k–£59,999",18,15.0
4,"£60k–£79,999",8,6.7
5,"£80k–£99,999",10,8.3
6,£100k+,10,8.3


5.3 Conditional aggregation

In [28]:
con.sql("""
SELECT
    job_family,

    COUNT(*) AS advertised_vacancies,

    SUM(
        CASE
            WHEN salary_midpoint < 40000
            THEN 1
            ELSE 0
        END
    ) AS under_40k,

    SUM(
        CASE
            WHEN salary_midpoint >= 40000
             AND salary_midpoint < 60000
            THEN 1
            ELSE 0
        END
    ) AS between_40k_60k,

    SUM(
        CASE
            WHEN salary_midpoint >= 60000
            THEN 1
            ELSE 0
        END
    ) AS salary_60k_plus,

    ROUND(
        MEDIAN(salary_midpoint),
        2
    ) AS median_salary

FROM core_jobs

WHERE salary_is_predicted = 0

GROUP BY job_family

ORDER BY advertised_vacancies DESC;
""").df()

,job_family,advertised_vacancies,under_40k,between_40k_60k,salary_60k_plus,median_salary
0,Data Analyst,82,28.0,31.0,23.0,45000.0
1,Business Intelligence,22,14.0,6.0,2.0,38728.5
2,Reporting Analytics,10,3.0,5.0,2.0,43500.0
3,Insights Analytics,5,1.0,3.0,1.0,50000.0
4,Management Information,1,0.0,1.0,0.0,50000.0


## 6. Advertiser Analysis

SQL aggregation and window functions are used to identify the most frequent
advertisers in the Core Analytics sample.

The `company` field may represent either a direct employer or a recruitment
intermediary, so results describe advertisement concentration rather than
definitive employer-level hiring activity.

In [29]:
con.sql("""
SELECT
    company,
    COUNT(*) AS vacancies,

    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct

FROM core_jobs

WHERE company IS NOT NULL

GROUP BY company

ORDER BY vacancies DESC, company

LIMIT 20;
""").df()

,company,vacancies,share_pct
0,Harnham - Data & Analytics Recruitment,8,3.3
1,Hypercreate Ltd,6,2.5
2,BAE Systems,4,1.6
3,Bright Purple Resourcing,3,1.2
4,Catalyst,3,1.2
5,Erin Associates,3,1.2
6,Proactive Appointments,3,1.2
7,Robert Half,3,1.2
8,The Progeny Group Limited,3,1.2
9,Velstar,3,1.2


6.1 Ranking com DENSE_RANK()

In [30]:
con.sql("""
WITH advertiser_counts AS (

    SELECT
        company,
        COUNT(*) AS vacancies

    FROM core_jobs

    WHERE company IS NOT NULL

    GROUP BY company
)

SELECT
    company,
    vacancies,

    DENSE_RANK() OVER (
        ORDER BY vacancies DESC
    ) AS advertiser_rank

FROM advertiser_counts

ORDER BY
    advertiser_rank,
    company

LIMIT 30;
""").df()

,company,vacancies,advertiser_rank
0,Harnham - Data & Analytics Recruitment,8,1
1,Hypercreate Ltd,6,2
2,BAE Systems,4,3
3,Bright Purple Resourcing,3,4
4,Catalyst,3,4
5,Erin Associates,3,4
6,Proactive Appointments,3,4
7,Robert Half,3,4
8,The Progeny Group Limited,3,4
9,Velstar,3,4


6.2 Top 5 e Top 10 share em uma única query

In [31]:
con.sql("""
WITH advertiser_counts AS (

    SELECT
        company,
        COUNT(*) AS vacancies

    FROM core_jobs

    WHERE company IS NOT NULL

    GROUP BY company
),

ranked_advertisers AS (

    SELECT
        company,
        vacancies,

        ROW_NUMBER() OVER (
            ORDER BY vacancies DESC, company
        ) AS advertiser_position

    FROM advertiser_counts
),

totals AS (

    SELECT
        COUNT(*) AS total_vacancies
    FROM core_jobs
)

SELECT
    ROUND(
        SUM(
            CASE
                WHEN advertiser_position <= 5
                THEN vacancies
                ELSE 0
            END
        )
        * 100.0
        / MAX(total_vacancies),
        1
    ) AS top_5_share_pct,

    ROUND(
        SUM(
            CASE
                WHEN advertiser_position <= 10
                THEN vacancies
                ELSE 0
            END
        )
        * 100.0
        / MAX(total_vacancies),
        1
    ) AS top_10_share_pct

FROM ranked_advertisers
CROSS JOIN totals;
""").df()

,top_5_share_pct,top_10_share_pct
0,9.8,16.0


## 7. JOIN Analysis — Search Query Overlap

The original collection process used multiple search terms to capture a
broader analytics labour-market sample.

The relationship between vacancies and collection queries was preserved in a
separate table because one vacancy may be returned by more than one search
term.

SQL joins are used here to analyse query coverage and overlap without
duplicating the canonical vacancy records.

In [32]:
con.sql("""
SELECT
    c.job_id,
    c.title,
    c.job_family,
    s.search_term

FROM core_jobs AS c

INNER JOIN job_search_terms AS s
    ON c.job_id = s.job_id

ORDER BY
    c.job_id,
    s.search_term

LIMIT 30;
""").df()

,job_id,title,job_family,search_term
0,4094618894,Data Analyst,Data Analyst,data analyst
1,4802504178,Business Intelligence Lead,Business Intelligence,business intelligence analyst
2,5119755250,Data and Reporting Analyst,Reporting Analytics,reporting analyst
3,5210480065,Data Analyst,Data Analyst,data analyst
4,5210480071,Data Analyst,Data Analyst,data analyst
5,5210480071,Data Analyst,Data Analyst,reporting analyst
6,5210480123,Lead Analyst - BI/DW - ETL & Reporting Developer,Business Intelligence,reporting analyst
7,5210489271,"Financial, Regulatory and Risk Reporting Analy...",Reporting Analytics,reporting analyst
8,5210489672,Data Analyst,Data Analyst,data analyst
9,5210489846,Senior Financial Reporting Analyst,Reporting Analytics,reporting analyst


In [33]:
con.sql("""
SELECT
    s.search_term,

    COUNT(DISTINCT c.job_id) AS core_vacancies,

    ROUND(
        COUNT(DISTINCT c.job_id) * 100.0
        / (
            SELECT COUNT(*)
            FROM core_jobs
        ),
        1
    ) AS core_coverage_pct

FROM core_jobs AS c

INNER JOIN job_search_terms AS s
    ON c.job_id = s.job_id

GROUP BY s.search_term

ORDER BY core_vacancies DESC;
""").df()

,search_term,core_vacancies,core_coverage_pct
0,data analyst,124,50.8
1,reporting analyst,77,31.6
2,business intelligence analyst,57,23.4


In [34]:
con.sql("""
SELECT
    c.job_id,
    c.title,
    c.company,
    c.job_family,

    COUNT(
        DISTINCT s.search_term
    ) AS query_matches,

    STRING_AGG(
        DISTINCT s.search_term,
        ' | '
    ) AS matched_queries

FROM core_jobs AS c

INNER JOIN job_search_terms AS s
    ON c.job_id = s.job_id

GROUP BY
    c.job_id,
    c.title,
    c.company,
    c.job_family

HAVING COUNT(
    DISTINCT s.search_term
) > 1

ORDER BY
    query_matches DESC,
    c.title;
""").df()

,job_id,title,company,job_family,query_matches,matched_queries
0,5814132404,Data Analyst,AWD online,Data Analyst,3,data analyst | business intelligence analyst |...
1,5846680046,Data Analyst,Pod Talent Ltd,Data Analyst,3,data analyst | business intelligence analyst |...
2,5814622826,Data Analyst,AWD Online,Data Analyst,3,data analyst | business intelligence analyst |...
3,5829818220,Business Intelligence Analyst,Xylem,Business Intelligence,2,business intelligence analyst | reporting analyst
4,5841480237,Data Analyst,Sportradar,Data Analyst,2,data analyst | business intelligence analyst
5,5804909149,Data Analyst,Lumen Partners,Data Analyst,2,data analyst | reporting analyst
6,5773388931,Data Analyst,GlobalData PLC,Data Analyst,2,data analyst | business intelligence analyst
7,5210480071,Data Analyst,wipro,Data Analyst,2,data analyst | reporting analyst
8,5808583587,Data Analyst,EXPRESS SOLICITORS,Data Analyst,2,data analyst | business intelligence analyst
9,5818755973,Data Analyst - Manufacturing,VANRATH,Data Analyst,2,data analyst | business intelligence analyst


In [35]:
con.sql("""
WITH query_matches AS (

    SELECT
        c.job_id,

        COUNT(
            DISTINCT s.search_term
        ) AS query_count

    FROM core_jobs AS c

    INNER JOIN job_search_terms AS s
        ON c.job_id = s.job_id

    GROUP BY c.job_id
)

SELECT
    query_count,
    COUNT(*) AS vacancies,

    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct

FROM query_matches

GROUP BY query_count

ORDER BY query_count;
""").df()

,query_count,vacancies,share_pct
0,1,233,95.5
1,2,8,3.3
2,3,3,1.2


## 8. Subqueries and Relative Salary Benchmarking

Individual advertised salaries are compared with the median salary of their
own job family.

This provides a relative benchmark that is more meaningful than comparing
every vacancy with a single market-wide salary threshold.

In [36]:
con.sql("""
SELECT
    c.job_id,
    c.title,
    c.company,
    c.job_family,

    ROUND(
        c.salary_midpoint,
        2
    ) AS salary_midpoint,

    ROUND(
        (
            SELECT MEDIAN(c2.salary_midpoint)

            FROM core_jobs AS c2

            WHERE
                c2.job_family = c.job_family
                AND c2.salary_is_predicted = 0
        ),
        2
    ) AS family_median_salary

FROM core_jobs AS c

WHERE
    c.salary_is_predicted = 0

    AND c.salary_midpoint >
        (
            SELECT MEDIAN(c3.salary_midpoint)

            FROM core_jobs AS c3

            WHERE
                c3.job_family = c.job_family
                AND c3.salary_is_predicted = 0
        )

ORDER BY
    c.job_family,
    c.salary_midpoint DESC;
""").df()

,job_id,title,company,job_family,salary_midpoint,family_median_salary
0,5843030453,Sr Business Intelligence Analyst - Commercial ...,Palo Alto Networks,Business Intelligence,80000.0,38728.5
1,5815905845,"Business Intelligence Analyst (Finance, SAP, P...",Eriban Business Services Ltd,Business Intelligence,60000.0,38728.5
2,5846995508,Lead BI Analyst - Tableau,Harnham - Data & Analytics Recruitment,Business Intelligence,56000.0,38728.5
3,5842064165,Lead Business Intelligence Analyst,Harnham - Data & Analytics Recruitment,Business Intelligence,55000.0,38728.5
4,5829404077,Business Intelligence Analyst,Datatech Analytics,Business Intelligence,50000.0,38728.5
5,5780350524,Finance Business Intelligence Analyst,Salutem Careers,Business Intelligence,45000.0,38728.5
6,5816031497,Finance Business Intelligence Analyst,Salutem,Business Intelligence,45000.0,38728.5
7,5812983061,Business Intelligence Analyst,Ecruit,Business Intelligence,40000.0,38728.5
8,5830781376,Business Intelligence Analyst,Police Scotland,Business Intelligence,39590.0,38728.5
9,5812613743,Business Intelligence Analyst,CIPS,Business Intelligence,39515.0,38728.5


In [37]:
con.sql("""
WITH family_benchmark AS (

    SELECT
        job_family,
        MEDIAN(salary_midpoint) AS family_median

    FROM core_jobs

    WHERE salary_is_predicted = 0

    GROUP BY job_family
)

SELECT
    c.job_family,

    COUNT(*) AS advertised_vacancies,

    SUM(
        CASE
            WHEN c.salary_midpoint > f.family_median
            THEN 1
            ELSE 0
        END
    ) AS above_family_median,

    ROUND(
        SUM(
            CASE
                WHEN c.salary_midpoint > f.family_median
                THEN 1
                ELSE 0
            END
        )
        * 100.0
        / COUNT(*),
        1
    ) AS above_median_pct

FROM core_jobs AS c

INNER JOIN family_benchmark AS f
    ON c.job_family = f.job_family

WHERE c.salary_is_predicted = 0

GROUP BY c.job_family

ORDER BY advertised_vacancies DESC;
""").df()

,job_family,advertised_vacancies,above_family_median,above_median_pct
0,Data Analyst,82,37.0,45.1
1,Business Intelligence,22,11.0,50.0
2,Reporting Analytics,10,5.0,50.0
3,Insights Analytics,5,2.0,40.0
4,Management Information,1,0.0,0.0


## 9. Salary Quartiles

Advertised Core Analytics salaries are divided into quartiles using the SQL
`NTILE()` window function.

The resulting salary segments are then used to examine whether particular job
families or geographic groups are disproportionately represented at the upper
or lower end of the observed salary distribution.

In [38]:
con.sql("""
SELECT
    job_id,
    title,
    company,
    job_family,
    salary_midpoint,

    NTILE(4) OVER (
        ORDER BY salary_midpoint
    ) AS salary_quartile

FROM core_jobs

WHERE salary_is_predicted = 0

ORDER BY salary_midpoint;
""").df()

,job_id,title,company,job_family,salary_midpoint,salary_quartile
0,5842554855,Junior Reporting Analyst,Work Force Nexus Recruitment Agency,Reporting Analytics,19200.0,1
1,5818222177,Data Analyst (AI Training),OneForma,Data Analyst,22880.0,1
2,5819806247,Trainee Data Analyst,Recruitment Solutions,Data Analyst,27000.0,1
3,5847071460,Data Insights & Business Intelligence Analyst,Alexander Mae Ltd,Business Intelligence,28000.0,1
4,5848067833,Data Insights & Business Intelligence Analyst,Alexander Mae Ltd,Business Intelligence,28000.0,1
...,...,...,...,...,...,...
115,5845203266,Data Analyst,Proactive Appointments,Data Analyst,113750.0,4
116,5826723271,Data Analyst,Alois Technologies Limited,Data Analyst,123500.0,4
117,5840340878,Data Analyst,TXP Technology x People,Data Analyst,124280.0,4
118,5840293769,Data Analyst,TXP,Data Analyst,124280.0,4


In [39]:
con.sql("""
WITH salary_quartiles AS (

    SELECT
        job_id,
        job_family,
        salary_midpoint,

        NTILE(4) OVER (
            ORDER BY salary_midpoint
        ) AS salary_quartile

    FROM core_jobs

    WHERE salary_is_predicted = 0
)

SELECT
    salary_quartile,

    COUNT(*) AS vacancies,

    ROUND(
        MIN(salary_midpoint),
        2
    ) AS min_salary,

    ROUND(
        MEDIAN(salary_midpoint),
        2
    ) AS median_salary,

    ROUND(
        MAX(salary_midpoint),
        2
    ) AS max_salary

FROM salary_quartiles

GROUP BY salary_quartile

ORDER BY salary_quartile;
""").df()

,salary_quartile,vacancies,min_salary,median_salary,max_salary
0,1,30,19200.0,32000.0,35000.0
1,2,30,35000.0,39552.5,44500.0
2,3,30,45000.0,50000.0,57000.0
3,4,30,57500.0,91750.0,136500.0


In [40]:
con.sql("""
WITH salary_quartiles AS (

    SELECT
        job_family,

        NTILE(4) OVER (
            ORDER BY salary_midpoint
        ) AS salary_quartile

    FROM core_jobs

    WHERE salary_is_predicted = 0
)

SELECT
    salary_quartile,
    job_family,

    COUNT(*) AS vacancies,

    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*))
            OVER (
                PARTITION BY salary_quartile
            ),
        1
    ) AS quartile_share_pct

FROM salary_quartiles

GROUP BY
    salary_quartile,
    job_family

ORDER BY
    salary_quartile,
    vacancies DESC;
""").df()

,salary_quartile,job_family,vacancies,quartile_share_pct
0,1,Data Analyst,21,70.0
1,1,Business Intelligence,8,26.7
2,1,Reporting Analytics,1,3.3
3,2,Data Analyst,17,56.7
4,2,Business Intelligence,7,23.3
5,2,Reporting Analytics,5,16.7
6,2,Insights Analytics,1,3.3
7,3,Data Analyst,21,70.0
8,3,Business Intelligence,5,16.7
9,3,Insights Analytics,3,10.0


## 10. Advertiser Posting Cadence

For advertisers associated with multiple Core Analytics vacancies, `LAG()` is
used to compare each advertisement with the previous observed posting from the
same organisation.

Because the Adzuna company field may represent recruitment intermediaries,
this is interpreted as advertisement cadence rather than direct employer
hiring cadence.

In [41]:
con.sql("""
WITH advertiser_jobs AS (

    SELECT
        job_id,
        company,
        title,

        TRY_CAST(
            created AS TIMESTAMPTZ
        ) AS created_at

    FROM core_jobs

    WHERE company IS NOT NULL
),

posting_sequence AS (

    SELECT
        job_id,
        company,
        title,
        created_at,

        LAG(created_at) OVER (
            PARTITION BY company
            ORDER BY created_at
        ) AS previous_posted_at

    FROM advertiser_jobs
)

SELECT
    company,
    title,
    created_at,
    previous_posted_at,

    DATE_DIFF(
        'day',
        previous_posted_at,
        created_at
    ) AS days_since_previous_post

FROM posting_sequence

WHERE previous_posted_at IS NOT NULL

ORDER BY
    company,
    created_at;
""").df()

,company,title,created_at,previous_posted_at,days_since_previous_post
0,BAE Systems,Lead Data Analyst/Project Controller,2026-08-13 15:06:21+00:00,2026-08-13 15:06:21+00:00,0
1,BAE Systems,Lead Data Analyst/Project Controller,2026-08-13 15:06:21+00:00,2026-08-13 15:06:21+00:00,0
2,BAE Systems,Lead Data Analyst/Project Controller,2026-08-13 21:23:17+00:00,2026-08-13 15:06:21+00:00,0
3,Baker Harding Limited,Data Analyst,2026-07-28 00:49:32+00:00,2026-07-16 08:42:00+00:00,12
4,Bright Purple Resourcing,Data Analyst - FTC,2026-07-30 00:49:44+00:00,2026-07-29 00:47:26+00:00,1
5,Bright Purple Resourcing,Data Analyst - FTC,2026-07-31 00:53:02+00:00,2026-07-30 00:49:44+00:00,1
6,CPR,Data Analyst,2026-08-13 15:51:07+00:00,2026-08-13 15:50:12+00:00,0
7,Catalyst,Data Analyst,2026-07-30 00:47:39+00:00,2026-07-29 20:55:12+00:00,1
8,Catalyst,Data Analyst,2026-08-13 11:37:22+00:00,2026-07-30 00:47:39+00:00,14
9,Cencora,Senior Analyst Business Intelligence - 3PL Ord...,2026-07-26 12:32:17+00:00,2026-07-26 12:24:30+00:00,0


## 11. SQL Key Findings

### Data integrity

SQL validation confirmed 390 unique classified vacancies, including 244
Core Analytics vacancies and 334 vacancies in the Extended Analytics market.

### Search-query coverage

The three collection queries contributed complementary coverage to the Core
Analytics dataset.

The "data analyst" query captured 124 Core vacancies (50.8%), followed by
"reporting analyst" with 77 (31.6%) and "business intelligence analyst" with
57 (23.4%).

Most Core vacancies (95.5%) appeared in only one search query, demonstrating
that the multi-query collection strategy expanded market coverage rather than
primarily retrieving the same advertisements repeatedly.

### Salary benchmarking

SQL reproduced the salary patterns identified in the Python workflow.

The median explicitly advertised salary is £44,750, compared with £49,106
for Adzuna-predicted salary values.

London-based vacancies show a median advertised salary of £52,500 compared
with £40,053 elsewhere in the UK, corresponding to an observed London
premium of 31.1%.

### Salary distribution

Using `NTILE(4)`, the 120 explicitly advertised salaries were divided into
four equal quartiles of 30 vacancies.

Median salaries increase from £32,000 in the first quartile to £91,750 in
the fourth quartile, highlighting substantial dispersion in the upper end of
the observed salary distribution.

### Advertiser structure

SQL ranking confirms that vacancy advertising is fragmented across many
organisations. The top five advertisers account for 9.8% of Core Analytics
vacancies and the top ten for 16.0%.

### SQL techniques demonstrated

The analysis applies:

- filtering and aggregation;
- CASE expressions;
- HAVING;
- Common Table Expressions (CTEs);
- scalar and correlated subqueries;
- INNER JOIN and CROSS JOIN;
- conditional aggregation;
- ROW_NUMBER, RANK and DENSE_RANK;
- NTILE;
- LAG;
- PARTITION BY;
- MEDIAN and other statistical aggregations.

In [42]:
import os

SQL_PATH = f"{PROJECT_PATH}/data/sql_outputs"

os.makedirs(
    SQL_PATH,
    exist_ok=True
)

In [43]:
sql_job_family_summary = con.sql("""
SELECT
    job_family,
    COUNT(*) AS vacancies,
    ROUND(
        COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER (),
        1
    ) AS share_pct
FROM core_jobs
GROUP BY job_family
ORDER BY vacancies DESC;
""").df()

sql_job_family_summary.to_csv(
    f"{SQL_PATH}/job_family_summary_sql.csv",
    index=False
)

In [44]:
sql_salary_summary = con.sql("""
SELECT
    salary_source,
    COUNT(*) AS vacancies,
    ROUND(AVG(salary_midpoint), 2) AS mean_salary,
    ROUND(MEDIAN(salary_midpoint), 2) AS median_salary,
    ROUND(MIN(salary_midpoint), 2) AS min_salary,
    ROUND(MAX(salary_midpoint), 2) AS max_salary
FROM core_jobs
GROUP BY salary_source;
""").df()

sql_salary_summary.to_csv(
    f"{SQL_PATH}/salary_summary_sql.csv",
    index=False
)

In [45]:
sql_search_coverage = con.sql("""
SELECT
    s.search_term,

    COUNT(DISTINCT c.job_id) AS core_vacancies,

    ROUND(
        COUNT(DISTINCT c.job_id) * 100.0
        / (
            SELECT COUNT(*)
            FROM core_jobs
        ),
        1
    ) AS core_coverage_pct

FROM core_jobs AS c

INNER JOIN job_search_terms AS s
    ON c.job_id = s.job_id

GROUP BY s.search_term

ORDER BY core_vacancies DESC;
""").df()

sql_search_coverage.to_csv(
    f"{SQL_PATH}/search_query_coverage_sql.csv",
    index=False
)

In [46]:
sql_salary_quartiles = con.sql("""
WITH salary_quartiles AS (

    SELECT
        salary_midpoint,

        NTILE(4) OVER (
            ORDER BY salary_midpoint
        ) AS salary_quartile

    FROM core_jobs

    WHERE salary_is_predicted = 0
)

SELECT
    salary_quartile,
    COUNT(*) AS vacancies,
    ROUND(MIN(salary_midpoint), 2) AS min_salary,
    ROUND(MEDIAN(salary_midpoint), 2) AS median_salary,
    ROUND(MAX(salary_midpoint), 2) AS max_salary

FROM salary_quartiles

GROUP BY salary_quartile

ORDER BY salary_quartile;
""").df()

sql_salary_quartiles.to_csv(
    f"{SQL_PATH}/salary_quartiles_sql.csv",
    index=False
)

In [47]:
print("SQL output files:")

for file in sorted(
    os.listdir(SQL_PATH)
):
    print(file)

SQL output files:
job_family_summary_sql.csv
salary_quartiles_sql.csv
salary_summary_sql.csv
search_query_coverage_sql.csv
